In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import json
import yaml
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CONFIG = ROOT / "configs/model/cv04_convnext_small_earlystop_gpu.yaml"

with open(CONFIG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

with open(ROOT / cfg["data"]["splits_json"], "r", encoding="utf-8") as f:
    splits = json.load(f)

train_raw = pd.read_csv(ROOT / cfg["data"]["train_df"])
val_raw = pd.read_csv(ROOT / cfg["data"]["val_df"])


def normalize_image_id_ext(value):
    text = str(value)
    if text.endswith(".0"):
        text = text[:-2]
    return text if Path(text).suffix else f"{text}.jpg"


def records_to_df(records):
    df = pd.DataFrame(records).copy()
    df["image_id_ext"] = df["image_id_ext"].map(normalize_image_id_ext)
    return df.reset_index(drop=True)


fold_frames = []
for fold_payload in splits["folds"]:
    df = records_to_df(fold_payload["records"])
    df["fold"] = int(fold_payload["fold"])
    fold_frames.append(df)

train_pool = pd.concat(fold_frames, ignore_index=True)
shadow_df = records_to_df(splits["shadow_holdout"]["records"])

for df in [train_raw, val_raw, train_pool, shadow_df]:
    df["_image_id_norm"] = df["image_id_ext"].map(normalize_image_id_ext)


def idset(df):
    return set(df["_image_id_norm"])


summary = pd.DataFrame(
    [
        {
            "source": "train_df.csv -> CV folds",
            "raw_rows": len(train_raw),
            "raw_unique_images": len(idset(train_raw)),
            "used_rows": len(train_pool),
            "used_unique_images": len(idset(train_pool)),
            "missing_unique_images": len(idset(train_raw) - idset(train_pool)),
            "extra_unique_images": len(idset(train_pool) - idset(train_raw)),
        },
        {
            "source": "val_df.csv -> shadow_holdout",
            "raw_rows": len(val_raw),
            "raw_unique_images": len(idset(val_raw)),
            "used_rows": len(shadow_df),
            "used_unique_images": len(idset(shadow_df)),
            "missing_unique_images": len(idset(val_raw) - idset(shadow_df)),
            "extra_unique_images": len(idset(shadow_df) - idset(val_raw)),
        },
    ]
)

display(summary)

,source,raw_rows,raw_unique_images,used_rows,used_unique_images,missing_unique_images,extra_unique_images
0,train_df.csv -> CV folds,4562,4562,4562,4562,0,0
1,val_df.csv -> shadow_holdout,500,500,477,477,23,0


In [3]:
def build_fold_frames(splits, fold):
    valid_records = splits["folds"][fold]["records"]
    train_records = []

    for fold_payload in splits["folds"]:
        if int(fold_payload["fold"]) != fold:
            train_records.extend(fold_payload["records"])

    return records_to_df(train_records), records_to_df(valid_records)


fold_stats = []

for fold in range(len(splits["folds"])):
    fold_train, fold_valid = build_fold_frames(splits, fold)

    fold_stats.append(
        {
            "fold": fold,
            "train_rows": len(fold_train),
            "valid_rows": len(fold_valid),
            "shadow_rows": len(shadow_df),
            "train_source": fold_train["source_dataset"].value_counts().to_dict(),
            "valid_source": fold_valid["source_dataset"].value_counts().to_dict(),
        }
    )

display(pd.DataFrame(fold_stats))

,fold,train_rows,valid_rows,shadow_rows,train_source,valid_source
0,0,3650,912,477,{'train_df': 3650},{'train_df': 912}
1,1,3650,912,477,{'train_df': 3650},{'train_df': 912}
2,2,3650,912,477,{'train_df': 3650},{'train_df': 912}
3,3,3648,914,477,{'train_df': 3648},{'train_df': 914}
4,4,3650,912,477,{'train_df': 3650},{'train_df': 912}


In [4]:
missing_val = val_raw[~val_raw["_image_id_norm"].isin(idset(shadow_df))]
display(missing_val)

,item_id,image,image_id_ext,result,label,ratio,_image_id_norm
63,1353607000180,http://labelimages.avito.ru/15335103603.jpg,15494054561,10,коридор / прихожая,1.000000,15494054561.jpg
83,1358510501269,http://labelimages.avito.ru/15518501005.jpg,15579494055,8,туалет,1.000000,15579494055.jpg
109,1306539500135,http://labelimages.avito.ru/14410885600.jpg,14419928610,15,подъезд / лестничная площадка,1.000000,14419928610.jpg
125,1347856002673,http://labelimages.avito.ru/15405276141.jpg,15332754747,6,детская,0.666667,15332754747.jpg
160,1295505751043,http://labelimages.avito.ru/15929277786.jpg,14417245929,15,подъезд / лестничная площадка,1.000000,14417245929.jpg
195,1287376251246,http://labelimages.avito.ru/15834968714.jpg,15620932559,6,детская,0.666667,15620932559.jpg
201,1361893250003,http://labelimages.avito.ru/15522309714.jpg,15662789316,17,предметы интерьера / быт.техника,1.000000,15662789316.jpg
231,1369268755559,http://labelimages.avito.ru/14254796425.jpg,15857997017,8,туалет,1.000000,15857997017.jpg
254,1295636750495,http://labelimages.avito.ru/14170696049.jpg,14409972018,8,туалет,0.666667,14409972018.jpg
274,1352079251084,http://labelimages.avito.ru/15422636354.jpg,15429903701,17,предметы интерьера / быт.техника,0.666667,15429903701.jpg


In [18]:
import json

s = json.load(open("../data/splits/splits_v1.json"))["summary"]
print(s["status_counts"])
print("val used:", s["shadow_holdout_rows_after_filters"], "/", s["val_df_raw_rows"])

{'train_df': {'ok': 4562}, 'val_df': {'missing': 23, 'ok': 477}}
val used: 477 / 500


In [20]:
import json                                                                                                                                                      
                                                                                                                                                                
s = json.load(open("../data/splits/splits_v1.json"))["summary"]                                                                                                  
print(s["status_counts"])                                                                                                                                        
print("val used:", s["shadow_holdout_rows_after_filters"], "/", s["val_df_raw_rows"]) 

{'train_df': {'ok': 4562}, 'val_df': {'ok': 500}}
val used: 500 / 500
